In [1]:
import pandas as pd
import numpy as np
import glob
import re

from tqdm import tqdm

import spacy
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

import language_tool_python

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anton\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
files = glob.glob("../data/raw/avis_*_traduit.xlsx")

df_list = []

for file in files:
    df_temp = pd.read_excel(file)
    df_list.append(df_temp)

df = pd.concat(df_list, ignore_index=True)

print("Shape:", df.shape)
df.head()

Shape: (34435, 11)


,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_cor,avis_cor_en
0,4.0,audurier-c-136272,La personne au téléphone était Clair et sympat...,L'olivier Assurance,auto,train,06/10/2021,01/10/2021,The person on the phone was clear and friendly...,NaN,NaN
1,4.0,paul-a-122970,"Satisfait.\n\nRéactivité, simplicité. Prix att...",APRIL Moto,moto,train,09/07/2021,01/07/2021,"Satisfied.\n\nReactivity, simplicity. Attracti...",NaN,NaN
2,1.0,kitty-38517,"Assureur à fuir, n assure pas ses responsabili...",SwissLife,vie,train,15/10/2020,01/10/2020,"Insurer to flee, does not ensure its responsib...",NaN,NaN
3,1.0,laure97134-87907,Voilà 3 mois que la GMF me fait attendre pour ...,GMF,habitation,train,03/03/2020,01/03/2020,The GMF has been waiting for a water damage fo...,NaN,NaN
4,3.0,bourouane-l-129916,Je suis bien avec cet assurance.elle est prati...,L'olivier Assurance,auto,train,28/08/2021,01/08/2021,I am good with this insurance. She is practica...,NaN,NaN


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34435 entries, 0 to 34434
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   note              24104 non-null  float64
 1   auteur            34434 non-null  object 
 2   avis              34435 non-null  object 
 3   assureur          34435 non-null  object 
 4   produit           34435 non-null  object 
 5   type              34435 non-null  object 
 6   date_publication  34435 non-null  object 
 7   date_exp          34435 non-null  object 
 8   avis_en           34433 non-null  object 
 9   avis_cor          435 non-null    object 
 10  avis_cor_en       431 non-null    object 
dtypes: float64(1), object(10)
memory usage: 2.9+ MB


In [12]:
df.isnull().sum()

note                10331
auteur                  1
avis                    0
assureur                0
produit                 0
type                    0
date_publication        0
date_exp                0
avis_en                 2
avis_cor            34000
avis_cor_en         34004
dtype: int64

In [14]:
df.nunique()

note                    5
auteur              33569
avis                34377
assureur               56
produit                13
type                    2
date_publication     1815
date_exp               61
avis_en             33264
avis_cor              435
avis_cor_en           431
dtype: int64

In [ ]:
df = df.dropna(subset=['avis', 'note']) # only for supervised learning
df['avis_en'] = df['avis_en'].fillna("")

In [16]:
df.isnull().sum()

note                    0
auteur                  1
avis                    0
assureur                0
produit                 0
type                    0
date_publication        0
date_exp                0
avis_en                 0
avis_cor            24104
avis_cor_en         24104
dtype: int64

In [ ]:
df['date_publication'] = pd.to_datetime(df['date_publication'], dayfirst=True, errors='coerce')

df['year'] = df['date_publication'].dt.year
df['month'] = df['date_publication'].dt.month

In [18]:
df.head()

,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_cor,avis_cor_en,year,month
0,4.0,audurier-c-136272,La personne au téléphone était Clair et sympat...,L'olivier Assurance,auto,train,2021-10-06,01/10/2021,The person on the phone was clear and friendly...,NaN,NaN,2021,10
1,4.0,paul-a-122970,"Satisfait.\n\nRéactivité, simplicité. Prix att...",APRIL Moto,moto,train,2021-07-09,01/07/2021,"Satisfied.\n\nReactivity, simplicity. Attracti...",NaN,NaN,2021,7
2,1.0,kitty-38517,"Assureur à fuir, n assure pas ses responsabili...",SwissLife,vie,train,2020-10-15,01/10/2020,"Insurer to flee, does not ensure its responsib...",NaN,NaN,2020,10
3,1.0,laure97134-87907,Voilà 3 mois que la GMF me fait attendre pour ...,GMF,habitation,train,2020-03-03,01/03/2020,The GMF has been waiting for a water damage fo...,NaN,NaN,2020,3
4,3.0,bourouane-l-129916,Je suis bien avec cet assurance.elle est prati...,L'olivier Assurance,auto,train,2021-08-28,01/08/2021,I am good with this insurance. She is practica...,NaN,NaN,2021,8


In [19]:
df = df.drop_duplicates(subset=['avis'])

python -m spacy download fr_core_news_sm
python -m spacy download en_core_web_sm

In [22]:
nlp_fr = spacy.load("fr_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

stop_fr = set(stopwords.words('french'))
stop_en = set(stopwords.words('english'))

In [23]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [24]:
def lemmatize(text, lang='fr'):
    doc = nlp_fr(text) if lang == 'fr' else nlp_en(text)
    return " ".join([token.lemma_ for token in doc if not token.is_stop])

In [29]:
from spellchecker import SpellChecker

spell = SpellChecker(language='fr')

def correct_text(text):
    words = text.split()
    corrected = [
        spell.correction(word) if spell.correction(word) else word
        for word in words
    ]
    return " ".join(corrected)

In [30]:
tqdm.pandas()

df['clean_text'] = df['avis'].progress_apply(clean_text)

# Optional (slow)
# df['corrected_text'] = df['clean_text'].progress_apply(correct_text)

df['lemmatized'] = df['clean_text'].progress_apply(lambda x: lemmatize(x, 'fr'))

100%|██████████| 24069/24069 [03:35<00:00, 111.51it/s]


In [31]:
df['text_length'] = df['avis'].apply(len)
df['word_count'] = df['avis'].apply(lambda x: len(str(x).split()))

In [ ]:
df.to_csv("avis.csv", index=False)

In [37]:
df.to_excel("avis.xlsx", index=False)

In [35]:
df.head()

,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_cor,avis_cor_en,year,month,clean_text,lemmatized,text_length,word_count
0,4.0,audurier-c-136272,La personne au téléphone était Clair et sympat...,L'olivier Assurance,auto,train,2021-10-06,01/10/2021,The person on the phone was clear and friendly...,NaN,NaN,2021,10,la personne au téléphone était clair et sympat...,téléphone clair sympathique bien expliquer rec...,167,26
1,4.0,paul-a-122970,"Satisfait.\n\nRéactivité, simplicité. Prix att...",APRIL Moto,moto,train,2021-07-09,01/07/2021,"Satisfied.\n\nReactivity, simplicity. Attracti...",NaN,NaN,2021,7,satisfait\n\nréactivité simplicité prix attrac...,satisfaire \n\n réactivité simplicité prix att...,163,22
2,1.0,kitty-38517,"Assureur à fuir, n assure pas ses responsabili...",SwissLife,vie,train,2020-10-15,01/10/2020,"Insurer to flee, does not ensure its responsib...",NaN,NaN,2020,10,assureur à fuir n assure pas ses responsabilit...,assureur fuir n assurer responsabilité agent d...,213,36
3,1.0,laure97134-87907,Voilà 3 mois que la GMF me fait attendre pour ...,GMF,habitation,train,2020-03-03,01/03/2020,The GMF has been waiting for a water damage fo...,NaN,NaN,2020,3,voilà 3 mois que la gmf me fait attendre pour ...,3 mois gmf attendre dégât eau jer contrat,124,23
4,3.0,bourouane-l-129916,Je suis bien avec cet assurance.elle est prati...,L'olivier Assurance,auto,train,2021-08-28,01/08/2021,I am good with this insurance. She is practica...,NaN,NaN,2021,8,je suis bien avec cet assuranceelle est pratiq...,bien assuranceelle pratique moin cherj trouve ...,170,28
